In [ ]:
!pip install -q ultralytics albumentations

In [ ]:
import torch, cv2, ultralytics
print("GPU:", torch.cuda.is_available(), "| cv2:", cv2.__version__)

In [ ]:
# 1. Force remove the old mount folder

# 2. Re-run the mount command
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
VIDEO = "/content/drive/MyDrive/staff_detection/copy_sample_video.mp4"

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11m.pt")

In [ ]:

model.track(source=VIDEO, classes=[0], tracker="botsort.yaml",
            conf=0.25, imgsz=960, save=True, project="/content/drive/MyDrive/staff_detection/runs", name="person_test")

In [ ]:
import cv2, os
BASE  = "/content/drive/MyDrive/staff_detection"
VIDEO = f"{BASE}/copy_sample_video.mp4"
OUT   = f"{BASE}/to_label"; os.makedirs(OUT, exist_ok=True)

WINDOWS = [(15.2,15.7),(16.9,17.5),(18.4,18.9),(20.3,20.5),(21.9,22.4),
           (32.4,32.8),(33.9,34.7),(35.7,36.2),(37.0,37.8),(38.9,39.4),
           (40.6,41.2),(42.4,43.0),(44.3,44.9),(46.2,46.8),(48.0,48.6)]
M = 0.3

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
saved = 0
for i in range(n):
    ok, f = cap.read()
    if not ok: break
    t = i / fps
    in_win = any(a-M <= t <= b+M for a, b in WINDOWS)
    if (in_win and i % 2 == 0) or (not in_win and i % 15 == 0):
        tag = "pos" if in_win else "neg"
        cv2.imwrite(f"{OUT}/{tag}_t{t:06.2f}_f{i:06d}.jpg", f)   # time and frame number in the name
        saved += 1
print("fps:", fps, "| saved:", saved)

In [ ]:
import glob
import os

# Target your main Google Drive root folder
target_folder = "/content/drive/MyDrive"

# Find all files starting with 'cardboard' in the root directory
cardboard_files = glob.glob(os.path.join(target_folder, "cardboard*"))

print(f"Found {len(cardboard_files)} files starting with 'cardboard' in MyDrive.")


In [ ]:
# Delete them safely
for file_path in cardboard_files:
  if os.path.isfile(file_path):
    os.remove(file_path)
    print(f"Deleted: {os.path.basename(file_path)}")

print("Cleanup complete!")

In [ ]:
!pip install roboflow


In [ ]:

from roboflow import Roboflow
rf = Roboflow(api_key="")
project = rf.workspace("").project("staff-name-tag-detection")
version = project.version(1)
dataset = version.download("yolov11")

In [ ]:
import glob, os
DS = "/content/Staff-Name-Tag-Detection-1"
print(open(f"{DS}/data.yaml").read())

labs = glob.glob(f"{DS}/train/labels/*.txt")
nonempty = [f for f in labs if os.path.getsize(f) > 0]
print("label files:", len(labs), "| with boxes:", len(nonempty), "| empty (negatives):", len(labs)-len(nonempty))

classes = set()
for f in nonempty:
    for line in open(f):
        classes.add(line.split()[0])
print("class ids used:", classes)      # you want only {'0'}

In [ ]:
import glob, os, re, shutil

SRC = "/content/Staff-Name-Tag-Detection-1"
DST = "/content/data"

VALID_W = [(38.9,39.4),(40.6,41.2),(42.4,43.0)]
TEST_W  = [(44.3,44.9),(46.2,46.8),(48.0,48.6)]
M = 0.3

def get_time(name):
    m = re.search(r"_t(\d+)[-.](\d+)_f(\d+)", name)
    return float(f"{m.group(1)}.{m.group(2)}") if m else None

def which(t):
    if t is None: return "train"
    for s, wins in [("valid", VALID_W), ("test", TEST_W)]:
        if any(a-M <= t <= b+M for a, b in wins): return s
    return "train"

for s in ["train", "valid", "test"]:
    os.makedirs(f"{DST}/{s}/images", exist_ok=True)
    os.makedirs(f"{DST}/{s}/labels", exist_ok=True)

counts = {"train": 0, "valid": 0, "test": 0}; unparsed = []
for img in glob.glob(f"{SRC}/train/images/*.jpg"):
    t = get_time(os.path.basename(img))
    if t is None: unparsed.append(os.path.basename(img))
    s = which(t)
    lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    shutil.copy(img, f"{DST}/{s}/images/")
    if os.path.exists(lab): shutil.copy(lab, f"{DST}/{s}/labels/")
    counts[s] += 1
print(counts); print("no time found:", unparsed)

In [ ]:
import cv2, numpy as np
ws, hs = [], []
for f in glob.glob("/content/data/train/labels/*.txt"):
    img = cv2.imread(f.replace("labels", "images").replace(".txt", ".jpg"))
    if img is None: continue
    H, W = img.shape[:2]
    for line in open(f):
        _, xc, yc, w, h = map(float, line.split())
        ws.append(w*W); hs.append(h*H)
print("frame size:", W, "x", H)
print("tag width px  5%/50%/95%:", np.percentile(ws, [5, 50, 95]).round(1))
print("tag height px 5%/50%/95%:", np.percentile(hs, [5, 50, 95]).round(1))

In [ ]:
import cv2, numpy as np, random, os, glob, shutil
from ultralytics import YOLO


In [ ]:
# ================= SETTINGS =================
BASE  = "/content/drive/MyDrive/staff_detection"
VIDEO = f"{BASE}/copy_sample_video.mp4"
DATA  = "/content/data"                 # your split folder (train/valid/test)
TAG_W_RANGE = (8, 32)                   # from your measured 11/17/22.5 px
TRAIN_END_S = 37.8                      # backgrounds only before this (valid/test stay unseen)
VARIANTS    = 3                         # synthetic images per background frame
STEP        = 3                         # use every 3rd frame

WINDOWS = [(15.2,15.7),(16.9,17.5),(18.4,18.9),(20.3,20.5),(21.9,22.4),
           (32.4,32.8),(33.9,34.7),(35.7,36.2),(37.0,37.8),(38.9,39.4),
           (40.6,41.2),(42.4,43.0),(44.3,44.9),(46.2,46.8),(48.0,48.6)]
MARGIN = 0.3                            # skip frames where the REAL tag is visible

IMG_DIR = f"{DATA}/train/images"
LAB_DIR = f"{DATA}/train/labels"
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(LAB_DIR, exist_ok=True)

# remove old synthetic files so re-running doesn't duplicate
for f in glob.glob(f"{IMG_DIR}/syn_*") + glob.glob(f"{LAB_DIR}/syn_*"):
    os.remove(f)


In [ ]:
# ================= TAG LIBRARY (RGBA cut-outs) =================
LIB = {}
for kind in ["clear", "mild", "heavy"]:
    imgs = [cv2.imread(f, cv2.IMREAD_UNCHANGED) for f in glob.glob(f"{BASE}/tags/{kind}/*.png")]
    LIB[kind] = [t for t in imgs if t is not None and t.ndim == 3 and t.shape[2] == 4]
print("tags loaded:", {k: len(v) for k, v in LIB.items()})

SOURCE_W = {"clear": 0.25, "mild": 0.20, "heavy": 0.55}
# tags are tiny (8-32 px), so no level-3 blur
BLUR_POLICY  = {"clear": ([0, 1, 2], [2, 3, 3]), "mild": ([0, 1, 2], [3, 3, 1]), "heavy": ([0], [1])}
PERSP_JITTER = {"clear": 0.12, "mild": 0.08, "heavy": 0.04}

In [ ]:
# ================= BLUR UTILS =================
def motion_kernel(k, angle):
  ker = np.zeros((k, k), np.float32)
  ker[k // 2, :] = 1
  M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), angle, 1)
  ker = cv2.warpAffine(ker, M, (k, k))
  return ker / max(ker.sum(), 1e-6)


def degrade(roi, level):
  if level == 0:
    return roi
  h, w = roi.shape[:2]
  if level == 1:
    return cv2.GaussianBlur(roi, (3, 3), 0)
  if random.random() < 0.5:
    return cv2.filter2D(
        roi,
        -1,
        motion_kernel(random.choice([3, 5, 7]), random.uniform(0, 180)),
    )
  small = cv2.resize(roi, (max(2, int(w * 0.6)), max(2, int(h * 0.6))))
  return cv2.resize(small, (w, h))

In [ ]:
# ================= TAG TRANSFORM =================
def make_tag_patch(tag, width_px, kind):
  h, w = tag.shape[:2]
  s = width_px / w
  t = cv2.resize(tag, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
  th, tw = t.shape[:2]
  if th < 3 or tw < 3:
    return None
  pad = int(max(th, tw) * 0.6) + 2
  t = cv2.copyMakeBorder(t, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0, 0))
  H, W = t.shape[:2]
  j = PERSP_JITTER[kind] * min(tw, th)
  src = np.float32([[pad, pad], [pad + tw, pad], [pad + tw, pad + th], [pad, pad + th]])
  dst = src + np.random.uniform(-j, j, src.shape).astype(np.float32)
  t = cv2.warpPerspective(t, cv2.getPerspectiveTransform(src, dst), (W, H))
  M = cv2.getRotationMatrix2D((W / 2, H / 2), random.uniform(0, 360), 1.0)
  t = cv2.warpAffine(t, M, (W, H))
  rgb = (
      t[..., :3].astype(np.float32) * random.uniform(0.6, 1.3)
      + random.uniform(-25, 25)
  )
  t[..., :3] = np.clip(rgb, 0, 255).astype(np.uint8)
  return t

In [ ]:
# ================= BODY CORE MASKS =================
seg_model = YOLO("yolo11m-seg.pt")


def get_person_masks(frame):
  H, W = frame.shape[:2]
  res = seg_model(
      frame, classes=[0], conf=0.25, imgsz=960, verbose=False
  )[0]
  out = []
  if res.masks is None:
    return out
  for poly in res.masks.xy:
    full = np.zeros((H, W), np.uint8)
    cv2.fillPoly(full, [poly.astype(np.int32)], 1)
    # Distance transform finds pixels deep inside the body, away from edges/arms
    dist = cv2.distanceTransform(full, cv2.DIST_L2, 3)
    if dist.max() < 4:
      continue
    core = (dist >= 0.55 * dist.max()).astype(
        np.uint8
    )  # deepest 45% = core/torso/back
    out.append((full, core))
  return out

In [ ]:
# ================= PASTE ONE TAG (CORE SAMPLING) =================
MIN_ON_PERSON = 0.85


def synth(frame, masks, motion):
  H, W = frame.shape[:2]
  kind = random.choices(list(SOURCE_W), weights=list(SOURCE_W.values()))[0]
  if not LIB[kind]:
    return None
  tag = random.choice(LIB[kind])
  patch = make_tag_patch(tag, random.randint(*TAG_W_RANGE), kind)
  if patch is None:
    return None
  ph, pw = patch.shape[:2]

  # Favor moving people using motion weights
  weights_list = [0.15 + (motion[f > 0].mean() * 10) for f, c in masks]
  full, core = random.choices(masks, weights=weights_list)[0]

  # Sample specifically from the body core (torso/back), avoiding hands/edges
  ys_m, xs_m = np.nonzero(core)
  if len(xs_m) == 0:
    return None
  k = random.randrange(len(xs_m))
  cx, cy = xs_m[k], ys_m[k]
  px, py = int(cx - pw / 2), int(cy - ph / 2)
  if px < 0 or py < 0 or px + pw > W or py + ph > H:
    return None

  fp = patch[..., 3] > 40
  if fp.sum() == 0:
    return None
  on_person = (full[py : py + ph, px : px + pw][fp] > 0).mean()
  if on_person < MIN_ON_PERSON:
    return None

  alpha = (
      cv2.GaussianBlur(patch[..., 3], (3, 3), 0).astype(np.float32)[..., None]
      / 255
  )
  out = frame.copy()
  roi = out[py : py + ph, px : px + pw].astype(np.float32)
  out[py : py + ph, px : px + pw] = (
      patch[..., :3] * alpha + roi * (1 - alpha)
  ).astype(np.uint8)

  ys, xs = np.where(fp)
  bx1, by1, bx2, by2 = px + xs.min(), py + ys.min(), px + xs.max(), py + ys.max()
  if (bx2 - bx1) < 4 or (by2 - by1) < 3:
    return None

  levels, weights = BLUR_POLICY[kind]
  level = random.choices(levels, weights=weights)[0]
  m = 3
  ry1, ry2, rx1, rx2 = (
      max(0, by1 - m),
      min(H, by2 + m),
      max(0, bx1 - m),
      min(W, bx2 + m),
  )
  out[ry1:ry2, rx1:rx2] = degrade(out[ry1:ry2, rx1:rx2], level)

  return (
      out,
      f"0 {(bx1+bx2)/2/W:.6f} {(by1+by2)/2/H:.6f} {(bx2-bx1)/W:.6f} {(by2-by1)/H:.6f}",
  )

In [ ]:
# ================= RUN OVER TRAIN PART OF VIDEO =================
for f in glob.glob(f"{IMG_DIR}/syn_*") + glob.glob(f"{LAB_DIR}/syn_*"):
  os.path.remove(f)

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)
SKIP = [(a - MARGIN, b + MARGIN) for a, b in WINDOWS]
saved, no_person, i = 0, 0, -1
prev = None

while True:
  ok, frame = cap.read()
  if not ok:
    break
  i += 1
  t = i / FPS

  # Track motion continuously across all frames (even skipped ones)
  gray = cv2.GaussianBlur(
      cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0
  )
  motion = (
      np.zeros_like(gray)
      if prev is None
      else (cv2.absdiff(gray, prev) > 18).astype(np.uint8)
  )
  prev = gray

  if t >= TRAIN_END_S:
    break
  if i % STEP:
    continue
  if any(a <= t <= b for a, b in SKIP):
    continue

  masks = get_person_masks(frame)
  if not masks:
    no_person += 1
    continue

  made = 0
  for _ in range(VARIANTS * 4):
    if made >= VARIANTS:
      break
    r = synth(frame, masks, motion)
    if r is None:
      continue
    img, label = r
    name = f"syn_{i:06d}_{saved}"
    cv2.imwrite(f"{IMG_DIR}/{name}.jpg", img)
    open(f"{LAB_DIR}/{name}.txt", "w").write(label)
    saved += 1
    made += 1

print(
    f"synthetic images saved: {saved} | frames with no person found: {no_person}"
)

# Backup to Drive
os.makedirs(f"{BASE}/synth_backup", exist_ok=True)
for f in glob.glob(f"{IMG_DIR}/syn_*"):
  shutil.copy(f, f"{BASE}/synth_backup/")

In [ ]:
import matplotlib.pyplot as plt, random, glob, cv2
IMG_DIR = "/content/data/train/images"
files = random.sample(glob.glob(f"{IMG_DIR}/syn_*.jpg"), 20)
fig, ax = plt.subplots(4, 5, figsize=(22, 14))
for a, f in zip(ax.ravel(), files):
    img = cv2.imread(f); H, W = img.shape[:2]
    for l in open(f.replace("images", "labels").replace(".jpg", ".txt")):
        _, xc, yc, w, h = map(float, l.split())
        cv2.rectangle(img, (int((xc-w/2)*W)-8, int((yc-h/2)*H)-8),
                           (int((xc+w/2)*W)+8, int((yc+h/2)*H)+8), (0, 255, 0), 1)
    a.imshow(img[..., ::-1]); a.axis("off")
plt.tight_layout(); plt.show()

/////////////////////////////

In [ ]:
import matplotlib.pyplot as plt, random, glob, cv2
IMG_DIR = "/content/data/train/images"
files = random.sample(glob.glob(f"{IMG_DIR}/syn_*.jpg"), 20)
fig, ax = plt.subplots(4, 5, figsize=(22, 14))
for a, f in zip(ax.ravel(), files):
    img = cv2.imread(f); H, W = img.shape[:2]
    for l in open(f.replace("images", "labels").replace(".jpg", ".txt")):
        _, xc, yc, w, h = map(float, l.split())
        cv2.rectangle(img, (int((xc-w/2)*W)-8, int((yc-h/2)*H)-8),
                           (int((xc+w/2)*W)+8, int((yc+h/2)*H)+8), (0, 255, 0), 1)
    a.imshow(img[..., ::-1]); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import glob, os
labs = [f for f in glob.glob("/content/data/train/labels/*.txt")
        if "syn_" not in f and "dup" not in f]
nonempty = [f for f in labs if os.path.getsize(f) > 0]
cls = {l.split()[0] for f in nonempty for l in open(f)}
print("real label files:", len(labs), "| with boxes:", len(nonempty),
      "| empty:", len(labs) - len(nonempty), "| classes:", cls)
print(open("/content/Staff-Name-Tag-Detection-1/data.yaml").read())

In [ ]:
import glob, os, collections

c = collections.Counter()
pos_empty = []
for lab in glob.glob("/content/data/train/labels/*.txt"):
    b = os.path.basename(lab)
    if b.startswith(("syn_", "dup")): continue            # skip synthetic and duplicates
    kind = "pos" if b.startswith("pos_") else "neg" if b.startswith("neg_") else "other"
    has_box = os.path.getsize(lab) > 0
    c[(kind, "box" if has_box else "empty")] += 1
    if kind == "pos" and not has_box:
        pos_empty.append(b)

print(c)
print("pos_ frames with NO box:", len(pos_empty))
print(pos_empty[:20])

In [ ]:
for split in ["valid", "test"]:
    labs = glob.glob(f"/content/data/{split}/labels/*.txt")
    pos = sum(os.path.getsize(f) > 0 for f in labs)
    boxes = sum(len(open(f).read().strip().splitlines()) for f in labs if os.path.getsize(f) > 0)
    print(split, "| images:", len(labs), "| with tag:", pos, "| total boxes:", boxes)

In [ ]:
import shutil

# delete old duplicates first, so re-running doesn't multiply them again
for f in glob.glob("/content/data/train/images/dup*") + glob.glob("/content/data/train/labels/dup*"):
    os.remove(f)

for img in glob.glob("/content/data/train/images/*.jpg"):
    b = os.path.basename(img)
    if b.startswith("syn_"): continue
    lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    for k in (1, 2):                                   # 2 extra copies = 3x in total
        shutil.copy(img, f"/content/data/train/images/dup{k}_{b}")
        if os.path.exists(lab):
            shutil.copy(lab, f"/content/data/train/labels/dup{k}_{b.rsplit('.', 1)[0]}.txt")

n_img = len(glob.glob("/content/data/train/images/*.jpg"))
n_syn = len(glob.glob("/content/data/train/images/syn_*.jpg"))
print("train images:", n_img, "| synthetic:", n_syn, "| real (incl. copies):", n_img - n_syn)

In [ ]:
import glob, os, random, shutil

KEEP = 250
IMG = "/content/data/train/images"
LAB = "/content/data/train/labels"
SPARE = "/content/data/syn_spare"            # not deleted, just moved aside
os.makedirs(f"{SPARE}/images", exist_ok=True)
os.makedirs(f"{SPARE}/labels", exist_ok=True)

syn = sorted(glob.glob(f"{IMG}/syn_*.jpg"))
random.seed(0)                                # same result every run
random.shuffle(syn)
move = syn[KEEP:]                             # everything after the first 250

for img in move:
    b = os.path.basename(img)
    lab = f"{LAB}/{b.rsplit('.', 1)[0]}.txt"
    shutil.move(img, f"{SPARE}/images/{b}")
    if os.path.exists(lab):
        shutil.move(lab, f"{SPARE}/labels/{os.path.basename(lab)}")

n_syn  = len(glob.glob(f"{IMG}/syn_*.jpg"))
n_all  = len(glob.glob(f"{IMG}/*.jpg"))
print("synthetic:", n_syn, "| real incl. copies:", n_all - n_syn,
      "| real share: %.0f%%" % (100 * (n_all - n_syn) / n_all))

In [ ]:
open("/content/data/data.yaml", "w").write("""path: /content/data
train: train/images
val: valid/images
test: test/images
names:
  0: name_tag
""")

In [ ]:
for sp in ["train", "valid", "test"]:
    for sub in ["images", "labels"]:
        os.makedirs(f"/content/data_real/{sp}/{sub}", exist_ok=True)

for f in glob.glob("/content/data/train/images/*.jpg"):
    if os.path.basename(f).startswith("syn_"): continue          # leave synthetic out
    shutil.copy(f, "/content/data_real/train/images/")
    l = f.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if os.path.exists(l): shutil.copy(l, "/content/data_real/train/labels/")

for sp in ["valid", "test"]:                                     # same valid/test for both runs
    shutil.copytree(f"/content/data/{sp}", f"/content/data_real/{sp}", dirs_exist_ok=True)

open("/content/data_real/data.yaml", "w").write("""path: /content/data_real
train: train/images
val: valid/images
test: test/images
names:
  0: name_tag
""")

In [ ]:
from ultralytics import YOLO
P = "/content/drive/MyDrive/staff_detection/runs"
COMMON = dict(imgsz=960, epochs=80, batch=16, patience=25,
              flipud=0.5, fliplr=0.5, degrees=180, scale=0.4,
              mosaic=1.0, close_mosaic=15, hsv_v=0.4, project=P)

YOLO("yolo11s.pt").train(data="/content/data_real/data.yaml", name="A_real_only", **COMMON)
YOLO("yolo11s.pt").train(data="/content/data/data.yaml",      name="B_real_plus_synth", **COMMON)

In [ ]:
for name in ["A_real_only", "B_real_plus_synth"]:
    m = YOLO(f"{P}/{name}/weights/best.pt")
    for split in ["val", "test"]:
        r = m.val(data="/content/data/data.yaml", split=split, imgsz=960,
                  conf=0.001, verbose=False)
        print(f"{name:20s} {split:5s}  P={r.box.mp:.2f}  R={r.box.mr:.2f}  mAP50={r.box.map50:.2f}")

In [ ]:
import cv2
WINDOWS = [(15.2,15.7),(16.9,17.5),(18.4,18.9),(20.3,20.5),(21.9,22.4),
           (32.4,32.8),(33.9,34.7),(35.7,36.2),(37.0,37.8),(38.9,39.4),
           (40.6,41.2),(42.4,43.0),(44.3,44.9),(46.2,46.8),(48.0,48.6)]
MARGIN = 0.3
def in_window(t): return any(a - MARGIN <= t <= b + MARGIN for a, b in WINDOWS)

best = YOLO(f"{P}/B_real_plus_synth/weights/best.pt")     # or A, whichever won
cap = cv2.VideoCapture(VIDEO); FPS = cap.get(cv2.CAP_PROP_FPS)
inside, outside, i = [], [], 0
while True:
    ok, f = cap.read()
    if not ok: break
    r = best(f, conf=0.15, imgsz=960, verbose=False)[0]
    if len(r.boxes):
        t = i / FPS
        (inside if in_window(t) else outside).append((round(t, 2), round(float(r.boxes.conf.max()), 2)))
    i += 1

print("detections inside windows :", len(inside))
print("detections OUTSIDE windows:", len(outside))
print("outside samples:", outside[:40])

In [ ]:
import cv2, os
from ultralytics import YOLO

BASE  = "/content/drive/MyDrive/staff_detection"
VIDEO = f"{BASE}/copy_sample_video.mp4"
P     = f"{BASE}/runs"
OUT   = f"{BASE}/B_detections.mp4"           # saved on Drive, so it survives a disconnect
CONF  = 0.15                                 # same low threshold you used in Step 9

WINDOWS = [(15.2,15.7),(16.9,17.5),(18.4,18.9),(20.3,20.5),(21.9,22.4),
           (32.4,32.8),(33.9,34.7),(35.7,36.2),(37.0,37.8),(38.9,39.4),
           (40.6,41.2),(42.4,43.0),(44.3,44.9),(46.2,46.8),(48.0,48.6)]
def in_win(t): return any(a <= t <= b for a, b in WINDOWS)     # your exact notes, no margin

model = YOLO(f"{P}/B_real_plus_synth/weights/best.pt")
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
W, H = int(cap.get(3)), int(cap.get(4))
vw = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

i = 0
while True:
    ok, f = cap.read()
    if not ok: break
    t = i / fps
    r = model(f, conf=CONF, imgsz=960, verbose=False)[0]

    for b in r.boxes:
        x1, y1, x2, y2 = map(int, b.xyxy[0]); c = float(b.conf)
        col = (0, 255, 0) if c >= 0.5 else (0, 165, 255) if c >= 0.3 else (0, 0, 255)  # green / orange / red
        # padded box so the tiny tag stays visible inside it
        cv2.rectangle(f, (x1-8, y1-8), (x2+8, y2+8), col, 1)
        cv2.putText(f, f"{c:.2f}", (x1-8, y1-12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)

    # banner: time, frame, and whether you noted a tag in this moment
    label = "TAG EXPECTED" if in_win(t) else "no tag expected"
    cv2.rectangle(f, (0, 0), (330, 26), (0, 0, 0), -1)
    cv2.putText(f, f"t={t:5.2f}s  frame {i}  {label}", (5, 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    vw.write(f); i += 1

cap.release(); vw.release()
print("saved:", OUT)

In [ ]:
from google.colab import files
files.download(OUT)

In [ ]:
import cv2, csv, os, numpy as np
from ultralytics import YOLO

BASE  = "/content/drive/MyDrive/staff_detection"
VIDEO = f"{BASE}/copy_sample_video.mp4"
TAG_W = f"{BASE}/runs/B_real_plus_synth/weights/best.pt"
OUT_V = f"{BASE}/B_tracking.mp4"
OUT_C = f"{BASE}/staff_frames.csv"
SHOTS = f"{BASE}/shots"; os.makedirs(SHOTS, exist_ok=True)

In [ ]:
STRONG      = 0.62    # tag confidence needed to count
MAX_JUMP    = 120     # px: how far the staff can move between "lost" and a new ID (short gaps)
JUMP_WINDOW = 50      # frames (~2 s): within this time, use position to re-link a changed ID
SIM_THRESH  = 0.75    # appearance similarity needed to re-link after leaving (0-1)
SIM_MARGIN  = 0.08    # best candidate must beat the second best by this much
CONFIRM     = 5       # frames the same candidate must hold before re-linking
SHOT_GAP    = 25      # min frames between screenshots

tag_model    = YOLO(TAG_W)
person_model = YOLO("yolo11m.pt")

GREEN, YELLOW = (0, 255, 0), (0, 255, 255)


In [ ]:
# ---------- helpers ----------
def hist_of(frame, box):
    """Colour fingerprint of the middle of a person's box (avoids background at the edges)."""
    x1, y1, x2, y2 = map(int, box); w, h = x2 - x1, y2 - y1
    crop = frame[max(0, y1 + int(.2*h)):y2 - int(.2*h), max(0, x1 + int(.2*w)):x2 - int(.2*w)]
    if crop.shape[0] < 4 or crop.shape[1] < 4: return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [12, 4, 4], [0, 180, 0, 256, 0, 256]).flatten()
    return (hist / (hist.sum() + 1e-9)).astype(np.float32)

def similarity(h1, h2):
    return 1.0 - cv2.compareHist(h1.reshape(-1, 1), h2.reshape(-1, 1), cv2.HISTCMP_BHATTACHARYYA)

def assign(tag_box, persons):
    """Which person does this tag belong to? (box contains the tag; if several, nearest centre)"""
    cx, cy = (tag_box[0] + tag_box[2]) / 2, (tag_box[1] + tag_box[3]) / 2
    best, bd = None, 1e9
    for tid, b in persons:
        if b[0] <= cx <= b[2] and b[1] <= cy <= b[3]:
            d = np.hypot(cx - (b[0] + b[2]) / 2, cy - (b[1] + b[3]) / 2)
            if d < bd: best, bd = tid, d
    return best

In [ ]:
# ---------- video ----------
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS); W, H = int(cap.get(3)), int(cap.get(4))
vw = cv2.VideoWriter(OUT_V, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
csvf = open(OUT_C, "w", newline=""); cw = csv.writer(csvf)
cw.writerow(["frame", "time_s", "staff_present", "state", "track_id", "cx", "cy",
             "x1", "y1", "x2", "y2", "tag_conf"])

In [ ]:
# ---------- staff memory ----------
staff_tid, staff_hist, last_center = None, None, None
mode = None            # "confirmed" (tag-verified) or "relinked" (appearance guess)
lost = 0
pend_tid, pend_n = None, 0
last_shot = -9999
n_present = 0
i = -1

while True:
    ok, f = cap.read()
    if not ok: break
    i += 1; t = i / fps
    raw = f.copy()                                   # clean copy for fingerprints and screenshots

    # 1) tag detections (strong only)
    tr = tag_model(f, conf=STRONG, imgsz=960, verbose=False)[0]
    tags = [(b.xyxy[0].cpu().numpy(), float(b.conf)) for b in tr.boxes]

    # 2) people + track IDs
    pr = person_model.track(f, persist=True, classes=[0], conf=0.25, imgsz=960,
                            tracker="bytetrack.yaml", verbose=False)[0]
    persons = []
    if pr.boxes.id is not None:
        persons = list(zip(pr.boxes.id.int().cpu().tolist(), pr.boxes.xyxy.cpu().numpy()))
    pd = dict(persons)

    # 3) a strong tag on a person -> that person is the staff
    tag_on_staff, tag_conf_now = False, 0.0
    for tb, c in tags:
        tid = assign(tb, persons)
        if tid is None: continue
        if tid != staff_tid: staff_hist = None       # new person: restart the fingerprint
        staff_tid, mode, lost, pend_tid, pend_n = tid, "confirmed", 0, None, 0
        tag_on_staff, tag_conf_now = True, max(tag_conf_now, c)
        h = hist_of(raw, pd[tid])
        if h is not None: staff_hist = h if staff_hist is None else 0.8 * staff_hist + 0.2 * h

    # 4) staff not visible -> try to find them again
    if staff_tid is not None and staff_tid not in pd:
        lost += 1
        cands = []
        if staff_hist is not None:
            for tid, b in persons:
                if lost <= JUMP_WINDOW and last_center is not None:        # short gap: ID probably switched
                    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
                    if np.hypot(cx - last_center[0], cy - last_center[1]) > MAX_JUMP: continue
                h = hist_of(raw, b)
                if h is not None: cands.append((similarity(staff_hist, h), tid))
        cands.sort(reverse=True)
        if cands and cands[0][0] >= SIM_THRESH and (len(cands) == 1 or cands[0][0] - cands[1][0] >= SIM_MARGIN):
            tid = cands[0][1]
            pend_n = pend_n + 1 if tid == pend_tid else 1
            pend_tid = tid
            if pend_n >= CONFIRM:
                staff_tid, mode, lost, pend_tid, pend_n = tid, "relinked", 0, None, 0
        else:
            pend_tid, pend_n = None, 0

    # 5) draw and log
    present = staff_tid is not None and staff_tid in pd
    state, row = "none", [i, round(t, 3), 0, "", "", "", "", "", "", "", "", ""]
    if present:
        x1, y1, x2, y2 = map(int, pd[staff_tid]); cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        last_center = (cx, cy); n_present += 1
        state = "tag" if tag_on_staff else ("tracked" if mode == "confirmed" else "relinked")
        col = GREEN if mode == "confirmed" else YELLOW
        txt = f"STAFF ({cx},{cy})" if mode == "confirmed" else f"STAFF? ({cx},{cy})"
        cv2.rectangle(f, (x1, y1), (x2, y2), col, 2)
        cv2.putText(f, txt, (x1, max(12, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        row = [i, round(t, 3), 1, state, staff_tid, cx, cy, x1, y1, x2, y2, round(tag_conf_now, 3)]
    elif staff_tid is not None:
        state = "lost"; row[3] = "lost"

    for tb, c in tags:                               # small green tag boxes (padded so the tag stays visible)
        x1, y1, x2, y2 = map(int, tb)
        cv2.rectangle(f, (x1 - 8, y1 - 8), (x2 + 8, y2 + 8), GREEN, 1)
        cv2.putText(f, f"tag {c:.2f}", (x1 - 8, y1 - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.4, GREEN, 1)
        if i - last_shot >= SHOT_GAP:
            cv2.imwrite(f"{SHOTS}/tag_f{i:06d}.jpg", f); last_shot = i

    cv2.rectangle(f, (0, 0), (360, 26), (0, 0, 0), -1)
    cv2.putText(f, f"t={t:5.2f}s  frame {i}  staff: {state}", (5, 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cw.writerow(row); vw.write(f)

In [ ]:

cap.release(); vw.release(); csvf.close()
print("frames with staff present:", n_present, "of", i + 1)
print("video:", OUT_V, "\ncsv:", OUT_C, "\nscreenshots:", SHOTS)